# Registro del proyecto ECU911: volumen de emergencias a t+3
Regresión exploratoria del número de emergencias por cantón y día. Conserva la división temporal de Semana 10, con separación de tres días. No evalúa todavía las alertas de demanda alta a t+3/t+7 del proyecto final.


In [1]:
import hashlib
import time
import platform
from pathlib import Path
import mlflow
import mlflow.sklearn
import pandas as pd
import sklearn
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from threadpoolctl import threadpool_limits
from pipeline_proyecto import RUTA_DATOS, cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
datos = pd.read_csv(RUTA_DATOS, parse_dates=["fecha"])
fecha_corte = datos.loc[X_prueba.index, "fecha"].min().date().isoformat()
huella = hashlib.sha256(Path(RUTA_DATOS).read_bytes()).hexdigest()
EXPERIMENTO = "ecu911-volumen-emergencias-t3"
mlflow.set_experiment(EXPERIMENTO)
print("Tracking:", mlflow.get_tracking_uri())
print("Train:", X_ent.shape, "Prueba:", X_prueba.shape, "Corte:", fecha_corte)


2026/11/04 20:30:29 INFO mlflow.tracking.fluent: Experiment with name 'ecu911-volumen-emergencias-t3' does not exist. Creating a new experiment.


Tracking: http://127.0.0.1:5050
Train: (337120, 11) Prueba: (84448, 11) Corte: 2025-08-17


In [2]:
modelos = [
    ("linea base mediana", DummyRegressor(strategy="median")),
    ("ridge alpha=10", Ridge(alpha=10.0, solver="lsqr")),
    ("bosque regularizado", RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=20, random_state=42, n_jobs=2)),
    ("red neuronal", TransformedTargetRegressor(regressor=MLPRegressor(hidden_layer_sizes=(64,), max_iter=300, batch_size=1024, early_stopping=True, validation_fraction=0.1, n_iter_no_change=15, random_state=42), transformer=StandardScaler())),
]
run_ids = []
for nombre, modelo in modelos:
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre) as run:
        run_ids.append(run.info.run_id)
        mlflow.log_params({**modelo.get_params(deep=True), "fecha_corte": fecha_corte,
            "separacion_dias": 3, "huella_sha256": huella, "filas_entrenamiento": len(X_ent),
            "filas_prueba": len(X_prueba), "objetivo": "obj_n_emergencias_t3",
            "python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__})
        mlflow.set_tags({"tipo": "regresion exploratoria", "division": "temporal por fechas completas"})
        inicio = time.perf_counter()
        with threadpool_limits(limits=2):
            pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        metricas = {"mae_entrenamiento": mean_absolute_error(y_ent, pipeline.predict(X_ent)),
                    "mae_prueba": mean_absolute_error(y_prueba, pipeline.predict(X_prueba)), "segundos": segundos}
        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(pipeline, "modelo")
        mlflow.log_artifact("pipeline_proyecto.py", artifact_path="codigo")
        print(nombre, run.info.run_id, metricas, flush=True)


2026/11/04 20:30:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


linea base mediana 6053744a4aa34f3894f61b263dfc1428 {'mae_entrenamiento': 38.96013882297105, 'mae_prueba': 36.630103732474424, 'segundos': 0.24934370801202022}


2026/11/04 20:30:31 INFO mlflow.tracking._tracking_service.client: 🏃 View run linea base mediana at: http://127.0.0.1:5050/#/experiments/3/runs/6053744a4aa34f3894f61b263dfc1428.


2026/11/04 20:30:31 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5050/#/experiments/3.


2026/11/04 20:30:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


ridge alpha=10 aa6975fa042c428f94438f4df744b7d6 {'mae_entrenamiento': 7.701378466863926, 'mae_prueba': 7.3365821483735445, 'segundos': 0.32600566701148637}


... (6 outputs left out)

2026/11/04 20:32:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


red neuronal d7c923b2b9694720b27a6e2556fdf58b {'mae_entrenamiento': 6.096481096793265, 'mae_prueba': 5.94719642447332, 'segundos': 12.113419839006383}


2026/11/04 20:32:24 INFO mlflow.tracking._tracking_service.client: 🏃 View run red neuronal at: http://127.0.0.1:5050/#/experiments/3/runs/d7c923b2b9694720b27a6e2556fdf58b.


2026/11/04 20:32:24 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5050/#/experiments/3.


In [3]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
# Exportar exclusivamente la ejecución completa actual; no mezcla intentos anteriores.
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id])
corridas = corridas[corridas.run_id.isin(run_ids)].sort_values("metrics.mae_prueba").reset_index(drop=True)
mejor_run_id = corridas.loc[0, "run_id"]
cliente = mlflow.MlflowClient()
for run_id in run_ids:
    cliente.set_tag(run_id, "candidato", "final" if run_id == mejor_run_id else "comparado")
modelo_final = mlflow.sklearn.load_model(f"runs:/{mejor_run_id}/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
assert abs(mae_reproducido - corridas.loc[0, "metrics.mae_prueba"]) < 1e-9
corridas["tags.candidato"] = corridas.run_id.map(lambda r: "final" if r == mejor_run_id else "comparado")
corridas.to_csv("corridas_proyecto.csv", index=False)
print(corridas[["run_id", "tags.mlflow.runName", "metrics.mae_entrenamiento", "metrics.mae_prueba", "metrics.segundos"]].to_string(index=False))
print("OK: modelo recargado reproduce MAE", mae_reproducido)


                          run_id tags.mlflow.runName  metrics.mae_entrenamiento  metrics.mae_prueba  metrics.segundos
0c40681fa7d54d56a75a95cd8ee29f5e bosque regularizado                   5.505628            5.632802         95.746274
d7c923b2b9694720b27a6e2556fdf58b        red neuronal                   6.096481            5.947196         12.113420
aa6975fa042c428f94438f4df744b7d6      ridge alpha=10                   7.701378            7.336582          0.326006
6053744a4aa34f3894f61b263dfc1428  linea base mediana                  38.960139           36.630104          0.249344
OK: modelo recargado reproduce MAE 5.632801743210028
